# Phase 1 — Step 1: Data Collection
Thu thập ảnh từ 3 nguồn:
1. **ArtiFact dataset** → real + fake_sd + fake_gan
2. **MidJourney v6** → fake_mj
3. **Flux.1 HF API** → fake_flux (UNSEEN generator)

**Chạy từng cell theo thứ tự. Nếu crash, chạy lại từ cell bị lỗi — không mất progress.**

## ⚙️ Cell 1 — Config
**Sửa đường dẫn tại đây nếu dataset mount khác tên.**

In [1]:
from pathlib import Path

# ── Output dirs ──────────────────────────────────────────────────────────────
RAW_DIR = Path('/kaggle/working/raw')
LOG_DIR = Path('/kaggle/working/logs')

# ── ArtiFact (awsaf49/artifact-dataset) ──────────────────────────────────────
ARTIFACT_BASE = Path('/kaggle/input/datasets/awsaf49/artifact-dataset')

ARTIFACT_SOURCES = {
    'real': [
        ARTIFACT_BASE / 'coco' / 'coco' / 'coco2017' / 'train2017',
        ARTIFACT_BASE / 'coco' / 'coco' / 'coco2017' / 'val2017',
    ],
    'fake_sd': [
        ARTIFACT_BASE / 'stable_diffusion' / 'stable' / 'images',
        # bỏ stable-face vì toàn face — bias nặng
    ],
    'fake_gan': [
        ARTIFACT_BASE / 'stylegan2' / 'car-part1' / 'car-512x384' / 'stylegan2-config-f-psi-0.5',
        ARTIFACT_BASE / 'stylegan2' / 'car-part2' / 'car-512x384' / 'stylegan2-config-f-psi-1.0',
        ARTIFACT_BASE / 'stylegan2' / 'church-part1' / 'church-256x256' / 'stylegan2-config-f-psi-0.5',
        ARTIFACT_BASE / 'stylegan2' / 'church-part2' / 'church-256x256' / 'stylegan2-config-f-psi-1.0',
        ARTIFACT_BASE / 'stylegan2' / 'horse-part1' / 'horse-256x256' / 'stylegan2-config-f-psi-0.5',
        ARTIFACT_BASE / 'stylegan2' / 'horse-part2' / 'horse-256x256' / 'stylegan2-config-f-psi-1.0',
        # bỏ ffhq và cat — toàn face/animal quá đặc thù
    ],
}

# ── MidJourney v6 (ivansivkovenin/midjourney-prompts-image-part1) ─────────────
MIDJOURNEY_DIR = Path('/kaggle/input/datasets/ivansivkovenin/midjourney-prompts-image-part1/images')

# ── Số lượng ảnh mỗi class ───────────────────────────────────────────────────
MIN_RESOLUTION = 200
TARGET_COUNTS = {
    'real':      8000,
    'fake_sd':   3000,
    'fake_gan':  2000,
    'fake_mj':   2000,
    'fake_flux':  109,   # UNSEEN — gen qua HF API
}

# ── Flux config ───────────────────────────────────────────────────────────────
FLUX_API_URL = 'https://router.huggingface.co/hf-inference/models/black-forest-labs/FLUX.1-schnell'
FLUX_SLEEP   = 2.0
FLUX_TIMEOUT = 120
FLUX_RETRY   = 3

FLUX_PROMPTS = [
    'a photo of a mountain landscape at golden hour',
    'a realistic photograph of a dense forest',
    'ocean waves crashing on rocky shore, photorealistic',
    'a serene lake reflecting clouds, documentary photography',
    'desert dunes at sunset, realistic photography',
    'a busy city street in the evening, realistic photo',
    'an old European cobblestone alley, overcast day',
    'a modern glass skyscraper reflecting the sky',
    'a traditional wooden house in rural area, realistic',
    'a crowded market scene, documentary photography',
    'a person walking in the rain, back view, street photography',
    'aerial view of people in a park, realistic photo',
    'silhouette of a person on a hill at sunset',
    'close-up photo of fresh fruits on wooden table',
    'a bowl of soup with steam rising, food photography',
    'a coffee cup on cafe table, morning light',
    'a vintage bicycle against a brick wall',
    'a dog running on a beach, action shot',
    'a cat sitting on a windowsill, natural light',
    'birds flying over a rice field, documentary',
    'lightning storm over city skyline, realistic',
    'snow-covered pine trees, winter photography',
    'a waterfall in a jungle, nature photography',
    'a harbor with fishing boats at dawn',
    'an empty road through autumn forest',
]
FLUX_STYLES = ['daytime', 'evening', 'morning', 'overcast', 'golden hour', 'night']

# ── Tạo thư mục output ────────────────────────────────────────────────────────
for label in TARGET_COUNTS:
    (RAW_DIR / label).mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print('✓ Config OK')
print(f'  RAW_DIR: {RAW_DIR}')
print(f'  Targets: {TARGET_COUNTS}')

✓ Config OK
  RAW_DIR: /kaggle/working/raw
  Targets: {'real': 8000, 'fake_sd': 3000, 'fake_gan': 2000, 'fake_mj': 2000, 'fake_flux': 109}


## 🔍 Cell 2 — Kiểm tra ArtiFact structure
Chạy cell này để xác nhận đường dẫn đúng trước khi collect.

In [2]:
import os

print('=== Datasets đã mount ===')
for d in sorted(os.listdir('/kaggle/input')):
    size = os.path.getsize(f'/kaggle/input/{d}') if os.path.isfile(f'/kaggle/input/{d}') else '(folder)'
    print(f'  /kaggle/input/{d}')

print()
print('=== ArtiFact folder detail ===')
if ARTIFACT_BASE.exists():
    for folder in sorted(ARTIFACT_BASE.iterdir()):
        if not folder.is_dir(): continue
        # Chỉ đếm 1 level — không rglob toàn bộ
        imgs = [f for f in folder.iterdir() 
                if f.suffix.lower() in ('.jpg','.png','.jpeg')]
        # Nếu ảnh nằm trong subfolder thì đếm subfolder
        subdirs = [f for f in folder.iterdir() if f.is_dir()]
        print(f'  {folder.name:30s} {len(imgs)} ảnh trực tiếp, {len(subdirs)} subfolder')
else:
    print(f'  [ERROR] {ARTIFACT_BASE} không tồn tại!')

=== Datasets đã mount ===
  /kaggle/input/datasets

=== ArtiFact folder detail ===
  afhq                           0 ảnh trực tiếp, 1 subfolder
  big_gan                        0 ảnh trực tiếp, 1 subfolder
  celebahq                       0 ảnh trực tiếp, 1 subfolder
  cips                           0 ảnh trực tiếp, 1 subfolder
  coco                           0 ảnh trực tiếp, 1 subfolder
  cycle_gan                      0 ảnh trực tiếp, 1 subfolder
  ddpm                           0 ảnh trực tiếp, 1 subfolder
  denoising_diffusion_gan        0 ảnh trực tiếp, 1 subfolder
  diffusion_gan                  0 ảnh trực tiếp, 1 subfolder
  face_synthetics                0 ảnh trực tiếp, 1 subfolder
  ffhq                           0 ảnh trực tiếp, 1 subfolder
  gansformer                     0 ảnh trực tiếp, 1 subfolder
  gau_gan                        0 ảnh trực tiếp, 1 subfolder
  generative_inpainting          0 ảnh trực tiếp, 2 subfolder
  glide                          0 ảnh trực tiếp,

In [3]:
for folder_name in ['coco', 'stable_diffusion', 'stylegan2']:
    folder = ARTIFACT_BASE / folder_name
    print(f'\n=== {folder_name} ===')
    for root, dirs, files in os.walk(folder):
        depth = root.replace(str(folder), '').count(os.sep)
        if depth > 3:
            continue
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(root)}/')


=== coco ===
coco/
  coco/
    coco2017/
      val2017/
      test2017/
      train2017/

=== stable_diffusion ===
stable_diffusion/
  stable-face/
    Female/
    Male/
  stable/
    images/

=== stylegan2 ===
stylegan2/
  ffhq-part1/
    ffhq-1024x1024/
      stylegan2-config-f-psi-0.5/
  car-part1/
    car-512x384/
      stylegan2-config-f-psi-0.5/
  horse-part2/
    horse-256x256/
      stylegan2-config-f-psi-1.0/
  ffhq-part2/
    ffhq-1024x1024/
      stylegan2-config-f-psi-1.0/
  church-part1/
    church-256x256/
      stylegan2-config-f-psi-0.5/
  church-part2/
    church-256x256/
      stylegan2-config-f-psi-1.0/
  car-part2/
    car-512x384/
      stylegan2-config-f-psi-1.0/
  cat-part1/
    cat-256x256/
      stylegan2-config-f-psi-0.5/
  horse-part1/
    horse-256x256/
      stylegan2-config-f-psi-0.5/
  cat-part2/
    cat-256x256/
      stylegan2-config-f-psi-1.0/


## 📦 Cell 3 — Helper functions

In [4]:
import random, shutil, json
from PIL import Image
from tqdm.notebook import tqdm

random.seed(42)
collection_log = {}

def is_valid_image(path: Path, min_size: int = MIN_RESOLUTION) -> bool:
    try:
        img = Image.open(path)
        w, h = img.size
        return w >= min_size and h >= min_size
    except Exception:
        return False

def collect_from_dirs(src_dirs: list, label: str, target: int) -> int:
    out_dir  = RAW_DIR / label
    existing = list(out_dir.glob('*.jpg')) + list(out_dir.glob('*.png'))
    count    = len(existing)

    if count >= target:
        print(f'  [{label}] Đã có {count} ảnh — skip')
        return count

    for src_dir in src_dirs:
        if count >= target:
            break
        src = Path(src_dir)
        if not src.exists():
            print(f'  [SKIP] {src} không tồn tại')
            continue

        all_imgs = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.webp']:
            all_imgs.extend(src.rglob(ext))
        random.shuffle(all_imgs)

        print(f'  Lấy từ {src.name} ({len(all_imgs):,} ảnh có sẵn)...')
        for img_path in tqdm(all_imgs, desc=f'  {label}', leave=False):
            if count >= target:
                break
            if not is_valid_image(img_path):
                continue
            ext = '.jpg' if img_path.suffix.lower() in ['.jpg','.jpeg','.webp'] else '.png'
            dst = out_dir / f'{label}_{count:05d}{ext}'
            try:
                shutil.copy2(img_path, dst)
                count += 1
            except Exception:
                continue

    print(f'  [{label}] ✓ {count}/{target}')
    return count

print('✓ Helper functions loaded')

✓ Helper functions loaded


## 🖼️ Cell 4 — Thu thập Real images (ArtiFact: coco + imagenet)

In [5]:
print('--- Real images ---')
n = collect_from_dirs(ARTIFACT_SOURCES['real'], 'real', TARGET_COUNTS['real'])
collection_log['real'] = n
print(f'Real: {n:,} ảnh')

--- Real images ---
  Lấy từ train2017 (118,202 ảnh có sẵn)...


  real:   0%|          | 0/118202 [00:00<?, ?it/s]

  [real] ✓ 8000/8000
Real: 8,000 ảnh


## 🎨 Cell 5 — Thu thập Fake SD (stable_diffusion + latent_diffusion + ddpm)

In [6]:
print('--- Fake SD ---')
n = collect_from_dirs(ARTIFACT_SOURCES['fake_sd'], 'fake_sd', TARGET_COUNTS['fake_sd'])
collection_log['fake_sd'] = n
print(f'Fake SD: {n:,} ảnh')

--- Fake SD ---
  Lấy từ images (19,000 ảnh có sẵn)...


  fake_sd:   0%|          | 0/19000 [00:00<?, ?it/s]

  [fake_sd] ✓ 3000/3000
Fake SD: 3,000 ảnh


## 🤖 Cell 6 — Thu thập Fake GAN (stylegan2 + pro_gan)

In [7]:
print('--- Fake GAN ---')
n = collect_from_dirs(ARTIFACT_SOURCES['fake_gan'], 'fake_gan', TARGET_COUNTS['fake_gan'])
collection_log['fake_gan'] = n
print(f'Fake GAN: {n:,} ảnh')

--- Fake GAN ---
  Lấy từ stylegan2-config-f-psi-0.5 (100,000 ảnh có sẵn)...


  fake_gan:   0%|          | 0/100000 [00:00<?, ?it/s]

  [fake_gan] ✓ 2000/2000
Fake GAN: 2,000 ảnh


## 🖌️ Cell 7 — Thu thập MidJourney v6

In [8]:
print('--- MidJourney v6 ---')
if not MIDJOURNEY_DIR.exists():
    print(f'[SKIP] {MIDJOURNEY_DIR} không tồn tại')
    collection_log['fake_mj'] = 0
else:
    n = collect_from_dirs([MIDJOURNEY_DIR], 'fake_mj', TARGET_COUNTS['fake_mj'])
    collection_log['fake_mj'] = n
    print(f'MidJourney: {n:,} ảnh')

--- MidJourney v6 ---
  Lấy từ images (127,980 ảnh có sẵn)...


  fake_mj:   0%|          | 0/127980 [00:00<?, ?it/s]

  [fake_mj] ✓ 2000/2000
MidJourney: 2,000 ảnh


## ⚡ Cell 8 — Gen Flux.1 (UNSEEN generator)
Cell này chạy lâu nhất (~40 phút). **Nếu crash, chạy lại cell này — sẽ tiếp tục từ ảnh cuối cùng.**

In [9]:
import requests, time
from kaggle_secrets import UserSecretsClient

try:
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('✓ HF token OK')
except Exception as e:
    print(f'[ERROR] Không lấy được HF_TOKEN: {e}')
    print('        Vào Kaggle Secrets → add HF_TOKEN')
    HF_TOKEN = None

if HF_TOKEN:
    out_dir  = RAW_DIR / 'fake_flux'
    target   = TARGET_COUNTS['fake_flux']
    existing = len(list(out_dir.glob('*.jpg')) + list(out_dir.glob('*.png')))

    print(f'Đã có {existing} ảnh, cần gen thêm {target - existing}')
    print(f'Ước tính: ~{int((target - existing) * (FLUX_SLEEP + 3) / 60)} phút')

    count  = existing
    errors = 0

    pbar = tqdm(total=target, initial=count, desc='Flux gen')

    while count < target:
        prompt      = random.choice(FLUX_PROMPTS)
        style       = random.choice(FLUX_STYLES)
        full_prompt = f'{prompt}, {style}, high quality'

        success = False
        for attempt in range(FLUX_RETRY):
            try:
                resp = requests.post(
                    FLUX_API_URL,
                    headers={'Authorization': f'Bearer {HF_TOKEN}',
                             'Content-Type': 'application/json'},
                    json={'inputs': full_prompt},
                    timeout=FLUX_TIMEOUT,
                )

                if resp.status_code == 200:
                    ext = '.jpg' if 'jpeg' in resp.headers.get('content-type','') else '.png'
                    (out_dir / f'flux_{count:05d}{ext}').write_bytes(resp.content)
                    (out_dir / f'flux_{count:05d}.txt').write_text(full_prompt)
                    count += 1
                    pbar.update(1)
                    success = True
                    break

                elif resp.status_code == 429:
                    wait = 30 * (attempt + 1)
                    tqdm.write(f'  Rate limit — chờ {wait}s...')
                    time.sleep(wait)

                elif resp.status_code == 503:
                    tqdm.write('  Model loading — chờ 20s...')
                    time.sleep(20)

                else:
                    tqdm.write(f'  HTTP {resp.status_code} — retry {attempt+1}')
                    time.sleep(5)

            except Exception as e:
                tqdm.write(f'  Exception: {e} — retry {attempt+1}')
                time.sleep(5)

        if not success:
            errors += 1
            if errors > 20:
                print('Quá nhiều lỗi — dừng lại')
                break

        time.sleep(FLUX_SLEEP)

    pbar.close()
    collection_log['fake_flux'] = count
    print(f'\n✓ Flux: {count} ảnh (lỗi: {errors})')

✓ HF token OK
Đã có 0 ảnh, cần gen thêm 109
Ước tính: ~9 phút


Flux gen:   0%|          | 0/109 [00:00<?, ?it/s]

  HTTP 402 — retry 1
  HTTP 402 — retry 2
  HTTP 402 — retry 3
  HTTP 402 — retry 1
  HTTP 402 — retry 2
  HTTP 402 — retry 3
  HTTP 402 — retry 1
  HTTP 402 — retry 2
  HTTP 402 — retry 3
  HTTP 402 — retry 1
  HTTP 402 — retry 2
  HTTP 402 — retry 3
  HTTP 402 — retry 1
  HTTP 402 — retry 2
  HTTP 402 — retry 3
  HTTP 402 — retry 1
  HTTP 402 — retry 2
  HTTP 402 — retry 3
  HTTP 402 — retry 1
  HTTP 402 — retry 2
  HTTP 402 — retry 3
  HTTP 402 — retry 1
  HTTP 402 — retry 2
  HTTP 402 — retry 3
  HTTP 402 — retry 1
  HTTP 402 — retry 2
  HTTP 402 — retry 3
  HTTP 402 — retry 1
  HTTP 402 — retry 2
  HTTP 402 — retry 3
  HTTP 402 — retry 1
  HTTP 402 — retry 2
  HTTP 402 — retry 3
  HTTP 402 — retry 1
  HTTP 402 — retry 2
  HTTP 402 — retry 3
  HTTP 402 — retry 1
  HTTP 402 — retry 2
  HTTP 402 — retry 3
  HTTP 402 — retry 1
  HTTP 402 — retry 2
  HTTP 402 — retry 3
  HTTP 402 — retry 1
  HTTP 402 — retry 2
  HTTP 402 — retry 3
  HTTP 402 — retry 1
  HTTP 402 — retry 2
  HTTP 402 — 

In [10]:
# Cell 8b — Bổ sung unseen generator bằng Kandinsky 3
import requests, time, random
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

KANDINSKY_URL = 'https://router.huggingface.co/hf-inference/models/stabilityai/stable-diffusion-3-medium-diffusers'

out_dir  = RAW_DIR / 'fake_flux'   # dùng chung folder
target   = 500
existing = len(list(out_dir.glob('*.jpg')) + list(out_dir.glob('*.png')))

print(f'Đã có {existing} ảnh (Flux), cần gen thêm {target - existing} (Kandinsky 3)')

count  = existing
errors = 0
pbar   = tqdm(total=target, initial=count, desc='Kandinsky gen')

while count < target:
    prompt      = random.choice(FLUX_PROMPTS)
    style       = random.choice(FLUX_STYLES)
    full_prompt = f'{prompt}, {style}, high quality'

    success = False
    for attempt in range(3):
        try:
            resp = requests.post(
                KANDINSKY_URL,
                headers={'Authorization': f'Bearer {HF_TOKEN}',
                         'Content-Type': 'application/json'},
                json={'inputs': full_prompt},
                timeout=120,
            )

            if resp.status_code == 200:
                ext = '.jpg' if 'jpeg' in resp.headers.get('content-type','') else '.png'
                (out_dir / f'kandinsky_{count:05d}{ext}').write_bytes(resp.content)
                (out_dir / f'kandinsky_{count:05d}.txt').write_text(full_prompt)
                count += 1
                pbar.update(1)
                success = True
                break

            elif resp.status_code == 402:
                print('  402 hết quota — thử ngày mai hoặc đổi model')
                break

            elif resp.status_code == 503:
                tqdm.write('  Model loading — chờ 20s...')
                time.sleep(20)

            else:
                tqdm.write(f'  HTTP {resp.status_code} — retry {attempt+1}')
                time.sleep(5)

        except Exception as e:
            tqdm.write(f'  Exception: {e} — retry {attempt+1}')
            time.sleep(5)

    if not success:
        errors += 1
        if errors > 10:
            print('Quá nhiều lỗi — dừng')
            break

    time.sleep(2.0)

pbar.close()
print(f'\n✓ Tổng unseen: {count} ảnh (Flux: 109, Kandinsky: {count-109})')
print(f'  Lỗi: {errors}')

Đã có 0 ảnh (Flux), cần gen thêm 500 (Kandinsky 3)


Kandinsky gen:   0%|          | 0/500 [00:00<?, ?it/s]

  402 hết quota — thử ngày mai hoặc đổi model
  402 hết quota — thử ngày mai hoặc đổi model
  402 hết quota — thử ngày mai hoặc đổi model
  402 hết quota — thử ngày mai hoặc đổi model
  402 hết quota — thử ngày mai hoặc đổi model
  402 hết quota — thử ngày mai hoặc đổi model
  402 hết quota — thử ngày mai hoặc đổi model
  402 hết quota — thử ngày mai hoặc đổi model
  402 hết quota — thử ngày mai hoặc đổi model
  402 hết quota — thử ngày mai hoặc đổi model
  402 hết quota — thử ngày mai hoặc đổi model
Quá nhiều lỗi — dừng

✓ Tổng unseen: 0 ảnh (Flux: 109, Kandinsky: -109)
  Lỗi: 11


## ✅ Cell 9 — Tổng kết & lưu log

In [11]:
print('=' * 50)
print('KẾT QUẢ STEP 1')
print('=' * 50)

total  = 0
all_ok = True

for label, target in TARGET_COUNTS.items():
    out_dir = RAW_DIR / label
    actual  = len(list(out_dir.glob('*.jpg')) + list(out_dir.glob('*.png')))
    pct     = actual / target * 100 if target > 0 else 0
    status  = '✓' if actual >= target * 0.8 else '⚠ thiếu'
    if actual < target * 0.8:
        all_ok = False
    print(f'  {status}  {label:15s} {actual:,}/{target:,} ({pct:.0f}%)')
    total += actual
    collection_log[label] = actual

print(f'\n  TOTAL: {total:,} ảnh')

# Lưu log
(LOG_DIR / 'step1_log.json').write_text(
    json.dumps(collection_log, indent=2)
)

if all_ok:
    print('\n✅ Step 1 hoàn thành — có thể chạy Step 2')
else:
    print('\n⚠  Một số class thiếu — kiểm tra lại config hoặc path dataset')

KẾT QUẢ STEP 1
  ✓  real            8,000/8,000 (100%)
  ✓  fake_sd         3,000/3,000 (100%)
  ✓  fake_gan        2,000/2,000 (100%)
  ✓  fake_mj         2,000/2,000 (100%)
  ⚠ thiếu  fake_flux       0/109 (0%)

  TOTAL: 15,000 ảnh

⚠  Một số class thiếu — kiểm tra lại config hoặc path dataset


In [12]:
import zipfile
from pathlib import Path
from tqdm.notebook import tqdm

RAW_DIR = Path('/kaggle/working/raw')
OUT_ZIP = Path('/kaggle/working/thesis_raw_images.zip')

all_files = [f for f in RAW_DIR.rglob('*') if f.is_file()]
print(f'Tổng file: {len(all_files):,}')

with zipfile.ZipFile(OUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in tqdm(all_files, desc='Nén'):
        zf.write(f, f.relative_to('/kaggle/working'))

print(f'✓ Xong: {OUT_ZIP.stat().st_size / 1024**3:.2f} GB')

Tổng file: 15,000


Nén:   0%|          | 0/15000 [00:00<?, ?it/s]

✓ Xong: 0.37 GB
